# ML Demand Forecasting with GBT

Generates configurable demand forecasts by store and product using a single
distributed Gradient-Boosted Trees model (Spark ML), tracked with Fabric-native MLflow.

## Data Flow
```
Configured Silver sales tables --> Feature Engineering --> GBT (distributed)
                                                              |
                                    +-------------------------+
                                    v                         v
                           Configured Gold forecast table  MLflow Experiment
```

## Model Details
- **Algorithm:** Gradient-Boosted Trees regression (Spark ML, fully distributed)
- **Approach:** Single global model for all store x product combinations
- **Features:** Lag sales (7/14/28d), rolling averages, day-of-week, month, store & product IDs
- **Forecast horizon:** Configurable recursive multi-step window
- **Tracking:** MLflow experiment with parameters, metrics, and logged model
- **Target metric:** Configurable MAPE threshold tracked across evaluated store x product combinations

## Usage
Schedule this notebook on the cadence that fits your refresh requirements.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.utils import AnalysisException
from pyspark.ml.linalg import Vectors, VectorUDT
from pyspark.ml.regression import GBTRegressor
from datetime import datetime, timezone, timedelta
import mlflow
import os

In [ ]:
# =============================================================================
# PARAMETERS
# =============================================================================

def get_env(var_name, default=None):
    return os.environ.get(var_name, default)

LAKEHOUSE_NAME = get_env("LAKEHOUSE_NAME", default="retail_lakehouse")
SILVER_DB = get_env("SILVER_DB", default="silver")
GOLD_DB = get_env("GOLD_DB", default="gold")
EXPERIMENT_NAME = get_env("MLFLOW_EXPERIMENT", default="demand_forecast")
RECEIPT_LINES_TABLE = get_env("RECEIPT_LINES_TABLE", default="fact_receipt_lines")
RECEIPTS_TABLE = get_env("RECEIPTS_TABLE", default="fact_receipts")
DEMAND_FORECAST_TABLE = get_env("DEMAND_FORECAST_TABLE", default="demand_forecast")
DEMAND_FORECAST_TABLE_NAME = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{DEMAND_FORECAST_TABLE}"

FORECAST_HORIZON_DAYS = int(get_env("FORECAST_HORIZON_DAYS", default="14"))
MIN_HISTORY_DAYS = int(get_env("MIN_HISTORY_DAYS", default="30"))
MIN_DAILY_SALES = float(get_env("MIN_DAILY_SALES", default="0.5"))
TARGET_MAPE = float(get_env("TARGET_MAPE", default="25.0"))
TARGET_COMBO_PCT = float(get_env("TARGET_COMBO_PCT", default="80.0"))
FORECAST_INTERVAL_Z = float(get_env("FORECAST_INTERVAL_Z", default="1.96"))



# GBT hyperparameters
MAX_ITER = 100
MAX_DEPTH = 6
STEP_SIZE = 0.1
SUBSAMPLING_RATE = 0.8

# Lag and rolling window sizes for feature engineering
LAG_DAYS = [1, 7, 14, 28]
ROLLING_WINDOWS = [7, 14, 28]

print(f"Configuration: SILVER_DB={SILVER_DB}, GOLD_DB={GOLD_DB}")
print(f"Source tables: {RECEIPTS_TABLE}, {RECEIPT_LINES_TABLE}")
print(f"Output table: {DEMAND_FORECAST_TABLE_NAME}")
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"Forecast horizon: {FORECAST_HORIZON_DAYS} days")
print(f"Target MAPE < {TARGET_MAPE}% for {TARGET_COMBO_PCT}% of evaluated combinations")
print(f"GBT: {MAX_ITER} trees, depth={MAX_DEPTH}, step={STEP_SIZE}")

In [ ]:
# =============================================================================
# HELPER FUNCTIONS & MLFLOW SETUP
# =============================================================================

def ensure_database(name):
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {LAKEHOUSE_NAME}.{name}")

def read_silver(table_name):
    return spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")

def silver_exists(table_name):
    try:
        spark.table(f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}")
        return True
    except AnalysisException:
        return False

ensure_database(GOLD_DB)

# Fabric auto-configures the MLflow tracking URI
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"MLflow experiment '{EXPERIMENT_NAME}' ready")

## Data Preparation

In [ ]:
if not silver_exists(RECEIPT_LINES_TABLE) or not silver_exists(RECEIPTS_TABLE):
    raise RuntimeError(
        f"Required tables {RECEIPT_LINES_TABLE} and {RECEIPTS_TABLE} not found in Silver layer"
    )

print(f"Reading {RECEIPT_LINES_TABLE} and joining with {RECEIPTS_TABLE}...")

df_lines = read_silver(RECEIPT_LINES_TABLE)
df_receipts = read_silver(RECEIPTS_TABLE)

# Aggregate to daily sales by store and product
df_daily_sales = (
    df_lines
    .join(df_receipts.select("receipt_id_ext", "store_id"), on="receipt_id_ext")
    .withColumn("sale_date", F.to_date("event_ts"))
    .groupBy("store_id", "product_id", "sale_date")
    .agg(
        F.sum("quantity").alias("units_sold"),
        F.sum("ext_price").alias("revenue"),
    )
)

total_daily_rows = df_daily_sales.count()
print(f"Daily sales rows: {total_daily_rows}")

# Filter to store-product combinations with sufficient history
df_history_check = (
    df_daily_sales
    .groupBy("store_id", "product_id")
    .agg(
        F.min("sale_date").alias("first_sale"),
        F.max("sale_date").alias("last_sale"),
        F.count("sale_date").alias("days_with_sales"),
        F.avg("units_sold").alias("avg_daily_sales"),
    )
    .withColumn(
        "history_days",
        F.datediff(F.col("last_sale"), F.col("first_sale")),
    )
    .filter(
        (F.col("history_days") >= MIN_HISTORY_DAYS)
        & (F.col("avg_daily_sales") >= MIN_DAILY_SALES)
    )
)

total_combinations = df_history_check.count()
print(f"Store-product combinations with sufficient history: {total_combinations}")

df_training_base = df_daily_sales.join(
    df_history_check.select("store_id", "product_id"),
    on=["store_id", "product_id"],
    how="inner",
)

print(f"Training base rows: {df_training_base.count()}")

## Feature Engineering

Build lag features, rolling averages, and calendar features from daily sales
data. All operations use Spark window functions — fully distributed.

In [ ]:
print("Building features...")

# Window for lag and rolling features, partitioned by store-product
w = Window.partitionBy("store_id", "product_id").orderBy("sale_date")

df_features = df_training_base

# Lag features: previous N days' sales
for lag in LAG_DAYS:
    df_features = df_features.withColumn(
        f"lag_{lag}d", F.lag("units_sold", lag).over(w)
    )

# Rolling average features
for win in ROLLING_WINDOWS:
    df_features = df_features.withColumn(
        f"rolling_avg_{win}d",
        F.avg("units_sold").over(w.rowsBetween(-win, -1)),
    )

# Rolling standard deviation (volatility)
df_features = df_features.withColumn(
    "rolling_std_14d",
    F.stddev("units_sold").over(w.rowsBetween(-14, -1)),
)

# Calendar features
df_features = (
    df_features
    .withColumn("day_of_week", F.dayofweek("sale_date"))
    .withColumn("day_of_month", F.dayofmonth("sale_date"))
    .withColumn("month", F.month("sale_date"))
    .withColumn("is_weekend", F.when(
        F.dayofweek("sale_date").isin(1, 7), 1
    ).otherwise(0))
    .withColumn("week_of_year", F.weekofyear("sale_date"))
)

# Drop rows where lag features are null (first N days per group)
df_features = df_features.filter(F.col(f"lag_{max(LAG_DAYS)}d").isNotNull())
df_features = df_features.na.fill(0.0)

feature_cols = (
    [f"lag_{d}d" for d in LAG_DAYS]
    + [f"rolling_avg_{w}d" for w in ROLLING_WINDOWS]
    + ["rolling_std_14d", "day_of_week", "day_of_month", "month",
       "is_weekend", "week_of_year", "store_id", "product_id"]
)

print(f"Features ({len(feature_cols)}): {', '.join(feature_cols)}")
print(f"Training rows after lag trimming: {df_features.count()}")

In [ ]:
# Hold out the most recent FORECAST_HORIZON_DAYS for evaluation
max_date = df_features.agg(F.max("sale_date")).first()[0]
cutoff_date = max_date - timedelta(days=FORECAST_HORIZON_DAYS)

df_train = df_features.filter(F.col("sale_date") <= cutoff_date)
df_test = df_features.filter(F.col("sale_date") > cutoff_date)

print(f"Max date: {max_date}")
print(f"Train/test cutoff: {cutoff_date}")
print(f"Train rows: {df_train.count()}")
print(f"Test rows: {df_test.count()}")

## Model Training & Evaluation

In [ ]:
# Build feature vectors without Spark ML slot metadata
to_dense_vector = F.udf(
    lambda *xs: Vectors.dense([0.0 if x is None else float(x) for x in xs]),
    VectorUDT(),
)

def _clean(df):
    return df.select(
        *[F.col(c).cast("double").alias(c) for c in feature_cols],
        F.col("units_sold").cast("double").alias("units_sold"),
        F.col("sale_date"),
    )

def _with_features(df):
    base = _clean(df)
    return base.select(
        "*",
        to_dense_vector(*[F.col(c) for c in feature_cols]).alias("features"),
    )

df_train_assembled = _with_features(df_train)
df_test_assembled = _with_features(df_test)

print("Training GBT model (distributed across Spark executors)...")

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="units_sold",
    predictionCol="predicted_units",
    maxIter=MAX_ITER,
    maxDepth=MAX_DEPTH,
    stepSize=STEP_SIZE,
    subsamplingRate=SUBSAMPLING_RATE,
    seed=42,
)

with mlflow.start_run(
    run_name=f"lgbm_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M')}"
) as run:
    mlflow.log_params({
        "algorithm": "spark_gbt_regressor",
        "forecast_horizon_days": FORECAST_HORIZON_DAYS,
        "min_history_days": MIN_HISTORY_DAYS,
        "min_daily_sales": MIN_DAILY_SALES,
        "target_mape": TARGET_MAPE,
        "max_iter": MAX_ITER,
        "max_depth": MAX_DEPTH,
        "step_size": STEP_SIZE,
        "subsampling_rate": SUBSAMPLING_RATE,
        "feature_count": len(feature_cols),
        "input_combinations": total_combinations,
        "train_rows": df_train.count(),
        "test_rows": df_test.count(),
    })

    model = gbt.fit(df_train_assembled.select("features", "units_sold"))

    # ── Evaluate on held-out test set ─────────────────────────────
    df_predictions = model.transform(df_test_assembled)
    df_predictions = df_predictions.withColumn(
        "predicted_units", F.greatest(F.col("predicted_units"), F.lit(0.0))
    )

    # Overall MAPE
    df_eval = df_predictions.filter(F.col("units_sold") > 0)
    overall_mape = df_eval.select(
        F.avg(
            F.abs(F.col("units_sold") - F.col("predicted_units"))
            / F.col("units_sold") * 100
        )
    ).first()[0] or 0.0

    # Per-combo MAPE for target analysis
    mape_by_combo = (
        df_eval
        .groupBy("store_id", "product_id")
        .agg(
            (F.avg(
                F.abs(F.col("units_sold") - F.col("predicted_units"))
                / F.col("units_sold")
            ) * 100).alias("avg_mape")
        )
    )

    stats = mape_by_combo.agg(
        F.avg("avg_mape").alias("mean_mape"),
        F.expr("percentile_approx(avg_mape, 0.5)").alias("median_mape"),
        F.expr("percentile_approx(avg_mape, 0.9)").alias("p90_mape"),
    ).first()

    evaluated_combos = mape_by_combo.count()
    meeting_target = mape_by_combo.filter(F.col("avg_mape") < TARGET_MAPE).count()
    pct_meeting = (
        meeting_target / evaluated_combos * 100 if evaluated_combos > 0 else 0.0
    )

    # RMSE
    rmse = df_eval.select(
        F.sqrt(F.avg(F.pow(F.col("units_sold") - F.col("predicted_units"), 2)))
    ).first()[0] or 0.0

    mlflow.log_metrics({
        "overall_mape": round(overall_mape, 2),
        "mean_combo_mape": round(float(stats["mean_mape"] or 0), 2),
        "median_combo_mape": round(float(stats["median_mape"] or 0), 2),
        "p90_combo_mape": round(float(stats["p90_mape"] or 0), 2),
        "rmse": round(rmse, 4),
        "pct_meeting_target": round(pct_meeting, 1),
        "evaluated_combinations": evaluated_combos,
    })

    # Log the model to MLflow
    mlflow.spark.log_model(model, "gbt_demand_model")

    print(f"MLflow run: {run.info.run_id}")
    print(f"\nTest-set evaluation ({evaluated_combos} store-product combos):")
    print(f"  Overall MAPE: {overall_mape:.1f}%")
    print(f"  RMSE: {rmse:.2f}")
    print(
        f"  Per-combo MAPE — mean: {stats['mean_mape']:.1f}%"
        f"  median: {stats['median_mape']:.1f}%"
        f"  p90: {stats['p90_mape']:.1f}%"
    )
    print(f"  Meeting target (<{TARGET_MAPE}%): {meeting_target} ({pct_meeting:.1f}%)")

    if pct_meeting >= TARGET_COMBO_PCT:
        print("\n  Target met")
    else:
        print("\n  Below target — consider tuning hyperparameters or adding regressors")

## Generate Forecasts

Use the trained model to predict the next 14 days. For multi-step forecasting,
we use the most recent actual data as the starting features for each
store x product combination.

In [ ]:
print("Generating 14-day forecasts for all qualifying store-product combinations...")

# Get the latest feature row per store-product (most recent day of actuals)
w_latest = Window.partitionBy("store_id", "product_id").orderBy(F.desc("sale_date"))

df_latest = (
    df_features
    .withColumn("_rn", F.row_number().over(w_latest))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

# For each forecast day, shift calendar features and predict
from functools import reduce

forecast_rows = []
for day_offset in range(1, FORECAST_HORIZON_DAYS + 1):
    df_forecast_day = (
        df_latest
        .withColumn("sale_date", F.date_add(F.col("sale_date"), day_offset))
        .withColumn("day_of_week", F.dayofweek("sale_date"))
        .withColumn("day_of_month", F.dayofmonth("sale_date"))
        .withColumn("month", F.month("sale_date"))
        .withColumn("is_weekend", F.when(
            F.dayofweek("sale_date").isin(1, 7), 1
        ).otherwise(0))
        .withColumn("week_of_year", F.weekofyear("sale_date"))
    )

    df_assembled = _with_features(df_forecast_day)
    df_pred = model.transform(df_assembled)
    df_pred = df_pred.withColumn(
        "predicted_units", F.greatest(F.col("predicted_units"), F.lit(0.0))
    )

    forecast_rows.append(
        df_pred.select(
            "store_id", "product_id",
            F.col("sale_date").alias("forecast_date"),
            "predicted_units",
        )
    )

df_all_forecasts = reduce(lambda a, b: a.unionByName(b), forecast_rows)

# Add confidence bounds (based on RMSE) and metadata
df_all_forecasts = (
    df_all_forecasts
    .withColumn("lower_bound", F.greatest(
        F.col("predicted_units") - F.lit(rmse * FORECAST_INTERVAL_Z), F.lit(0.0)
    ))
    .withColumn("upper_bound",
        F.col("predicted_units") + F.lit(rmse * FORECAST_INTERVAL_Z)
    )
    .withColumn("mape", F.lit(overall_mape))
    .withColumn("generated_at", F.lit(datetime.now(timezone.utc)))
)

# Cast to final schema
df_final = (
    df_all_forecasts
    .select(
        F.col("store_id").cast("long"),
        F.col("product_id").cast("long"),
        F.col("forecast_date").cast("date"),
        F.col("predicted_units").cast("double"),
        F.col("lower_bound").cast("double"),
        F.col("upper_bound").cast("double"),
        F.col("mape").cast("double"),
        F.col("generated_at").cast("timestamp"),
    )
)

# Save to Gold layer
table_name = DEMAND_FORECAST_TABLE_NAME
df_final.write.format("delta").mode("overwrite").saveAsTable(table_name)

forecast_count = spark.table(table_name).count()
combos_forecasted = (
    spark.table(table_name).select("store_id", "product_id").distinct().count()
)
print(f"Saved {forecast_count} forecast records to {table_name}")
print(f"Covering {combos_forecasted} store-product combinations")

In [ ]:
print("=" * 60)
print("DEMAND FORECASTING COMPLETE")
print("=" * 60)

gold_tables = spark.sql(f"SHOW TABLES IN {LAKEHOUSE_NAME}.{GOLD_DB}").collect()
print(f"\nGold layer ({GOLD_DB}): {len(gold_tables)} tables")
print(f"Forecast table: {DEMAND_FORECAST_TABLE_NAME}")
print(f"Forecast horizon: {FORECAST_HORIZON_DAYS} days")
print(f"\nMLflow experiment: {EXPERIMENT_NAME}")
print(f"Model logged: gbt_demand_model")
print("View results in Fabric > Experiments")
print("\nSchedule this notebook on your preferred cadence for fresh forecasts.")